<h1 style="color:DodgerBlue">Индивидуальный проект</h1>

**Студент:** Решетова Арина, группа СМАРТБ-25-1

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

Способы оплаты (PaymentMethod)

### Вариант задания 16


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

Создан базовый класс **PaymentMethod** (способ оплаты) с атрибутами PaymentMethodId, MethodName, MinAmount и методами ProcessPayment, CheckMinimumAmount, GetPaymentDetails. На его основе сделаны три производных класса:

- **OnlinePayment** - атрибут PaymentUrl, метод ProcessPayment переопределен (в оплату включается url платежной системы);
- **BankTransfer** - атрибут BankData, метод CheckMinimumAmount переопределен (проверка мин. суммы с учетом комиссии банка);
- **CashPayment** - атрибут CashPickupPoint, метод GetPaymentDetails переопределен (показывает место выдачи наличных).

#### Дополнительное задание
К базовому классу добавлены атрибуты IsActive, PaymentsCount, TotalProcessed и методы Activate, Deactivate, GetStatistics. В производные классы добавлены свои атрибуты (Currency, CommissionPercent, WorkingHours) и методы (IsSecureUrl, CalculateCommission, ChangePickupPoint). В проекте использованы коллекции `List<T>` и `Dictionary<TKey, TValue>`, делегат `PaymentHandler`, событие `PaymentProcessed` и лямбда-выражения.


<h2 style="color:DodgerBlue">Реализация:</h2>

----

Сначала описываем классы:

In [ ]:
using System;
using System.Collections.Generic;
using System.Linq;

// делегат для уведомл. об обработке платежа
public delegate void PaymentHandler(PaymentMethod method, decimal amount, bool success);

// базовый класс - способ оплаты
public class PaymentMethod
{
    public int PaymentMethodId { get; private set; }
    public string MethodName { get; set; }
    public decimal MinAmount { get; set; }

    // доп. атрибуты
    public bool IsActive { get; private set; }
    public int PaymentsCount { get; private set; }
    public decimal TotalProcessed { get; private set; }

    // событие - платеж обработан
    public event PaymentHandler PaymentProcessed;

    public PaymentMethod(int id, string name, decimal minAmount)
    {
        PaymentMethodId = id;
        MethodName = name;
        MinAmount = minAmount;
        IsActive = true;
    }

    // обработка платежа
    public virtual bool ProcessPayment(decimal amount)
    {
        if (!IsActive)
        {
            Console.WriteLine($"Способ \"{MethodName}\" отключен");
            PaymentProcessed?.Invoke(this, amount, false);
            return false;
        }

        if (!CheckMinimumAmount(amount))
        {
            Console.WriteLine($"Сумма {amount} слишком мала, платеж отклонен");
            PaymentProcessed?.Invoke(this, amount, false);
            return false;
        }

        PaymentsCount++;
        TotalProcessed += amount;
        Console.WriteLine($"Платеж на сумму {amount} через \"{MethodName}\" выполнен");
        PaymentProcessed?.Invoke(this, amount, true);
        return true;
    }

    // проверка мин. суммы
    public virtual bool CheckMinimumAmount(decimal amount)
    {
        return amount >= MinAmount;
    }

    // детали способа оплаты
    public virtual string GetPaymentDetails()
    {
        return $"ID: {PaymentMethodId}, способ: {MethodName}, мин. сумма: {MinAmount}";
    }

    // доп. методы
    public void Activate()
    {
        IsActive = true;
    }

    public void Deactivate()
    {
        IsActive = false;
    }

    public string GetStatistics()
    {
        return $"{MethodName}: платежей - {PaymentsCount}, всего - {TotalProcessed}";
    }
}

// онлайн оплата
public class OnlinePayment : PaymentMethod
{
    public string PaymentUrl { get; set; }
    public string Currency { get; set; }   // доп. атрибут

    public OnlinePayment(int id, string name, decimal minAmount, string url, string currency)
        : base(id, name, minAmount)
    {
        PaymentUrl = url;
        Currency = currency;
    }

    // переопределяем - добавляем url в процесс оплаты
    public override bool ProcessPayment(decimal amount)
    {
        Console.WriteLine($"Переход на платежную систему: {PaymentUrl}");
        return base.ProcessPayment(amount);
    }

    public override string GetPaymentDetails()
    {
        return base.GetPaymentDetails() + $", url: {PaymentUrl}, валюта: {Currency}";
    }

    // доп. метод
    public bool IsSecureUrl()
    {
        return PaymentUrl.StartsWith("https://");
    }
}

// банковский перевод
public class BankTransfer : PaymentMethod
{
    public string BankData { get; set; }
    public decimal CommissionPercent { get; set; }   // доп. атрибут, комиссия в %

    public BankTransfer(int id, string name, decimal minAmount, string bankData, decimal commissionPercent)
        : base(id, name, minAmount)
    {
        BankData = bankData;
        CommissionPercent = commissionPercent;
    }

    // доп. метод - считаем комиссию
    public decimal CalculateCommission(decimal amount)
    {
        return amount * CommissionPercent / 100;
    }

    // переопределяем - мин. сумма проверяется с учетом комиссии
    public override bool CheckMinimumAmount(decimal amount)
    {
        decimal afterCommission = amount - CalculateCommission(amount);
        return afterCommission >= MinAmount;
    }

    public override string GetPaymentDetails()
    {
        return base.GetPaymentDetails() + $", банк. данные: {BankData}, комиссия: {CommissionPercent}%";
    }
}

// наличные
public class CashPayment : PaymentMethod
{
    public string CashPickupPoint { get; set; }
    public string WorkingHours { get; set; }   // доп. атрибут

    public CashPayment(int id, string name, decimal minAmount, string pickupPoint, string workingHours)
        : base(id, name, minAmount)
    {
        CashPickupPoint = pickupPoint;
        WorkingHours = workingHours;
    }

    // доп. метод
    public void ChangePickupPoint(string newPoint)
    {
        CashPickupPoint = newPoint;
    }

    // переопределяем - показываем место выдачи
    public override string GetPaymentDetails()
    {
        return base.GetPaymentDetails() + $", место выдачи: {CashPickupPoint}, время работы: {WorkingHours}";
    }
}


Теперь проверяем работу программы:

In [ ]:
// обработчик события (метод)
static void OnPaymentProcessed(PaymentMethod method, decimal amount, bool success)
{
    string result = success ? "успешно" : "отклонен";
    Console.WriteLine($"  [Событие] {method.MethodName}: платеж {amount} - {result}");
}

// создаем способы оплаты
OnlinePayment online = new OnlinePayment(1, "Онлайн оплата", 100, "https://pay.example.com", "RUB");
BankTransfer bank = new BankTransfer(2, "Банковский перевод", 500, "Счет 40817810000000000001", 2);
CashPayment cash = new CashPayment(3, "Наличные", 50, "ул. Ленина, 10", "9:00-18:00");

// коллекция List
List<PaymentMethod> methods = new List<PaymentMethod> { online, bank, cash };

// коллекция Dictionary (ключ - id)
Dictionary<int, PaymentMethod> methodsById = new Dictionary<int, PaymentMethod>();
foreach (var m in methods)
{
    methodsById.Add(m.PaymentMethodId, m);
}

// подписываемся на событие
foreach (var m in methods)
{
    m.PaymentProcessed += OnPaymentProcessed;
}
// еще один подписчик - лямбда
bank.PaymentProcessed += (method, amount, success) =>
{
    if (!success) Console.WriteLine("  [Лямбда] Проверьте сумму перевода!");
};

// полиморфизм - один и тот же вызов, разное поведение
Console.WriteLine("=== Детали способов оплаты ===");
foreach (var m in methods)
{
    Console.WriteLine(m.GetPaymentDetails());
}
Console.WriteLine();

Console.WriteLine("=== Платежи на 520 ===");
foreach (var m in methods)
{
    m.ProcessPayment(520);
}
Console.WriteLine();

// банк: 510 - комиссия 2% = 499.8, меньше 500 -> отказ
Console.WriteLine("=== Проверка комиссии банка ===");
bank.ProcessPayment(510);
Console.WriteLine($"Комиссия с 1000: {bank.CalculateCommission(1000)}");
Console.WriteLine();

// отключаем способ
Console.WriteLine("=== Отключаем наличные ===");
cash.Deactivate();
cash.ProcessPayment(200);
cash.Activate();
Console.WriteLine();

// поиск в Dictionary
if (methodsById.TryGetValue(1, out PaymentMethod found))
{
    Console.WriteLine("Найден способ по id 1:");
    Console.WriteLine(found.GetPaymentDetails());
}
Console.WriteLine();

// лямбда-выражения
var cheap = methods.Where(m => m.MinAmount <= 100).ToList();
Console.WriteLine("Способы с мин. суммой до 100: " + string.Join(", ", cheap.Select(m => m.MethodName)));
Console.WriteLine($"Онлайн url безопасный? {online.IsSecureUrl()}");
Console.WriteLine();

// статистика
Console.WriteLine("=== Статистика ===");
foreach (var m in methods)
{
    Console.WriteLine(m.GetStatistics());
}
